In [ ]:

import numpy as np
import matplotlib.pyplot as plt


# ---------------------------------------------------------------
# 1. Dataset
# ---------------------------------------------------------------
def load_coffee_data():
    """
    Creates a synthetic coffee roasting dataset.
    A roast is "good" (y=1) if it falls inside a rough diagonal band:
    higher temperature needs shorter duration, and vice versa.
    """
    rng = np.random.default_rng(2)
    X = rng.random((400, 2))
    print(X)
    X[:, 0] = X[:, 0] * (285 - 150) + 150   # Temperature: 150-285 C
    X[:, 1] = X[:, 1] * 4 + 11.5            # Duration: 11.5-15.5 min
    Y = np.zeros(len(X))

    i = 0
    for t, d in X:
        y = -3 / (260 - 175) * t + 21.25
        if (t > 175 and t < 260 and d > 12 and d < 15 and d <= y):
            Y[i] = 1
        else:
            Y[i] = 0
        i += 1

    return X, Y.reshape(-1, 1)


def normalize(X):
    """Zero-mean, unit-variance normalization (like tf.keras Normalization)."""
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


# ---------------------------------------------------------------
# 2. Forward propagation in NumPy
# ---------------------------------------------------------------
g = sigmoid


def my_dense(a_in, W, b):
    units = W.shape[1]
    a_out = np.zeros(units)
    for j in range(units):
        w = W[:, j]
        z = np.dot(w, a_in) + b[j]
        a_out[j] = g(z)
    return a_out


def my_sequential(x, W1, b1, W2, b2):
    a1 = my_dense(x, W1, b1)
    a2 = my_dense(a1, W2, b2)
    return a2


def my_predict(X, W1, b1, W2, b2):
    m = X.shape[0]
    p = np.zeros((m, 1))
    for i in range(m):
        p[i, 0] = my_sequential(X[i], W1, b1, W2, b2)[0]
    return p


# ---------------------------------------------------------------
# 3. Trained weights (copied from the TensorFlow version of this lab)
# ---------------------------------------------------------------
W1 = np.array([[-8.93, 0.29, 12.9], [-0.1, -7.32, 10.81]])
b1 = np.array([-9.82, -9.28, 0.96])
W2 = np.array([[-31.18], [-27.59], [-32.56]])
b2 = np.array([15.41])


# ---------------------------------------------------------------
# 4. Run it
# ---------------------------------------------------------------
if __name__ == "__main__":
    X, Y = load_coffee_data()
    print(f"X shape: {X.shape}, Y shape: {Y.shape}")

    print(f"Temperature Max, Min pre normalization: {np.max(X[:,0]):0.2f}, {np.min(X[:,0]):0.2f}")
    print(f"Duration    Max, Min pre normalization: {np.max(X[:,1]):0.2f}, {np.min(X[:,1]):0.2f}")

    Xn = normalize(X)
    print(f"Temperature Max, Min post normalization: {np.max(Xn[:,0]):0.2f}, {np.min(Xn[:,0]):0.2f}")
    print(f"Duration    Max, Min post normalization: {np.max(Xn[:,1]):0.2f}, {np.min(Xn[:,1]):0.2f}")

    # Two test examples (unnormalized), same as the lab
    X_tst = np.array([
        [200, 13.9],  # expected: good roast
        [200, 17.0],  # expected: bad roast
    ])
    mu, sigma = np.mean(X, axis=0), np.std(X, axis=0)
    X_tstn = (X_tst - mu) / sigma

    predictions = my_predict(X_tstn, W1, b1, W2, b2)
    print(f"probabilities = \n{predictions}")

    yhat = (predictions >= 0.5).astype(int)
    print(f"decisions = \n{yhat}")

    # ------------------------------------------------------------
    # Visualize: decision regions over the training data
    # ------------------------------------------------------------
    def netf(x):
        xn = (x - mu) / sigma
        return my_predict(xn, W1, b1, W2, b2)

    xx, yy = np.meshgrid(
        np.linspace(150, 285, 100),
        np.linspace(11.5, 15.5, 100),
    )
    grid = np.c_[xx.ravel(), yy.ravel()]
    probs = netf(grid).reshape(xx.shape)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Raw probability output
    im0 = axes[0].contourf(xx, yy, probs, levels=20, cmap="RdBu_r", alpha=0.8)
    axes[0].scatter(X[Y[:, 0] == 1, 0], X[Y[:, 0] == 1, 1], marker="x", c="k", label="good (1)")
    axes[0].scatter(X[Y[:, 0] == 0, 0], X[Y[:, 0] == 0, 1], marker="o", edgecolors="k", facecolors="none", label="bad (0)")
    axes[0].set_title("Network output (probability)")
    axes[0].set_xlabel("Temperature (C)")
    axes[0].set_ylabel("Duration (min)")
    axes[0].legend()
    fig.colorbar(im0, ax=axes[0])

    # Thresholded decision
    decisions = (probs >= 0.5).astype(int)
    axes[1].contourf(xx, yy, decisions, levels=[-0.5, 0.5, 1.5], colors=["#f4a582", "#92c5de"], alpha=0.8)
    axes[1].scatter(X[Y[:, 0] == 1, 0], X[Y[:, 0] == 1, 1], marker="x", c="k", label="good (1)")
    axes[1].scatter(X[Y[:, 0] == 0, 0], X[Y[:, 0] == 0, 1], marker="o", edgecolors="k", facecolors="none", label="bad (0)")
    axes[1].set_title("Decision (threshold 0.5)")
    axes[1].set_xlabel("Temperature (C)")
    axes[1].set_ylabel("Duration (min)")
    axes[1].legend()

    plt.tight_layout()
    plt.savefig("coffee_roasting_result.png", dpi=120)
    print("Saved plot to coffee_roasting_result.png")